In [28]:
!pip install -q peft transformers datasets

In [29]:
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from datasets import load_dataset


In [30]:
model = TFAutoModel.from_pretrained("bert-base-multilingual-cased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [31]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

In [32]:
inputs = tokenizer(['Hello world', 'Hi how are you'], padding=True, truncation=True,
                  return_tensors='tf')
inputs


{'input_ids': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[  101, 31178, 11356,   102,     0,     0],
       [  101, 20065, 14796, 10301, 13028,   102]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[1, 1, 1, 1, 0, 0],
       [1, 1, 1, 1, 1, 1]], dtype=int32)>}

In [33]:
output = model(inputs)
output

TFBaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=<tf.Tensor: shape=(2, 6, 768), dtype=float32, numpy=
array([[[-1.28848299e-01, -8.29536080e-01,  1.08536017e+00, ...,
          5.10799401e-02, -2.50097305e-01, -3.54952753e-01],
        [-2.75410414e-01, -1.26741171e+00,  9.98244762e-01, ...,
          7.08758533e-02, -2.07902625e-01, -5.94257951e-01],
        [-2.89130330e-01, -1.45810485e+00,  1.17758656e+00, ...,
         -2.50020355e-01, -1.22904047e-01, -6.09883904e-01],
        [-5.04235439e-02, -1.12271714e+00,  1.45674062e+00, ...,
         -1.91058427e-01, -2.60178030e-01, -2.80654937e-01],
        [-5.48606962e-02, -9.83053982e-01,  1.10246348e+00, ...,
          1.66603737e-02, -2.27826670e-01, -2.97334790e-01],
        [-2.06222758e-02, -1.01458371e+00,  1.14159250e+00, ...,
          4.06113006e-02, -1.87947854e-01, -2.49661654e-01]],

       [[-5.10684438e-02, -6.24206364e-02,  6.69196963e-01, ...,
          8.03123116e-01, -5.63445449e-01,  1.05505258e-01]

In [34]:
dataset_name = "twitter_complaints"
dataset = load_dataset("ought/raft", dataset_name)
dataset["train"][0]
{"Tweet text": "@HMRCcustomers No this is my first job", "ID": 0, "Label": 2}

{'Tweet text': '@HMRCcustomers No this is my first job', 'ID': 0, 'Label': 2}

In [35]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Tweet text', 'ID', 'Label'],
        num_rows: 50
    })
    test: Dataset({
        features: ['Tweet text', 'ID', 'Label'],
        num_rows: 3399
    })
})

In [36]:
def tokenize(batch):
    return tokenizer(batch["Tweet text"], padding=True, truncation=True)

In [37]:
dataset_encoded = dataset.map(tokenize, batched=True, batch_size=None)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/3399 [00:00<?, ? examples/s]

In [38]:
dataset_encoded

DatasetDict({
    train: Dataset({
        features: ['Tweet text', 'ID', 'Label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50
    })
    test: Dataset({
        features: ['Tweet text', 'ID', 'Label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3399
    })
})

In [39]:
dataset_encoded.set_format('tf',columns=['input_ids', 'attention_mask', 'token_type_ids', 'Label'])
BATCH_SIZE = 64

def order(inp):
   data = list(inp.values())
   return {
        'input_ids': data[1],
        'attention_mask': data[2],
        'token_type_ids': data[3]
    }, data[0]
train_dataset = tf.data.Dataset.from_tensor_slices(dataset_encoded['train'][:])
train_dataset = train_dataset.batch(BATCH_SIZE).shuffle(1000)
train_dataset = train_dataset.map(order, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices(dataset_encoded['test'][:])
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.map(order, num_parallel_calls=tf.data.AUTOTUNE)

In [40]:
inp, out = next(iter(train_dataset)) # a batch from train_dataset
print(inp, '\n\n', out)

{'input_ids': <tf.Tensor: shape=(50, 68), dtype=int64, numpy=
array([[   101,    137, 109234, ...,      0,      0,      0],
       [   101,    137,  82258, ...,      0,      0,      0],
       [   101,  14535,    146, ...,      0,      0,      0],
       ...,
       [   101,    137,  19686, ...,      0,      0,      0],
       [   101,  25290,  11367, ...,      0,      0,      0],
       [   101,    137,  18875, ...,      0,      0,      0]])>, 'attention_mask': <tf.Tensor: shape=(50, 68), dtype=int64, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])>, 'token_type_ids': <tf.Tensor: shape=(50, 68), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])>} 

 tf

In [41]:
class BertForSequenceClassification(tf.keras.Model):

    def __init__(self, bert_model, num_classes):
        super().__init__()
        self.bert = bert_model
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs):
        x = self.bert(inputs)[1]
        return self.fc(x)

In [42]:
classifier = BertForSequenceClassification(model, num_classes=6)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [43]:
history = classifier.fit(
    train_dataset,
    epochs=15
)

Epoch 1/15
1/1 [==============================] - 114s 114s/step - loss: 1.7739 - accuracy: 0.0200
Epoch 2/15
1/1 [==============================] - 40s 40s/step - loss: 1.5087 - accuracy: 0.4600
Epoch 3/15
1/1 [==============================] - 44s 44s/step - loss: 1.3351 - accuracy: 0.4800
Epoch 4/15
1/1 [==============================] - 40s 40s/step - loss: 1.1364 - accuracy: 0.6800
Epoch 5/15
1/1 [==============================] - 36s 36s/step - loss: 0.9880 - accuracy: 0.6400
Epoch 6/15
1/1 [==============================] - 36s 36s/step - loss: 0.9279 - accuracy: 0.6600
Epoch 7/15
1/1 [==============================] - 38s 38s/step - loss: 0.9042 - accuracy: 0.6600
Epoch 8/15
1/1 [==============================] - 39s 39s/step - loss: 0.8185 - accuracy: 0.6600
Epoch 9/15
1/1 [==============================] - 36s 36s/step - loss: 0.7529 - accuracy: 0.6600
Epoch 10/15
1/1 [==============================] - 37s 37s/step - loss: 0.7060 - accuracy: 0.6400
Epoch 11/15
1/1 [==========

In [44]:
classifier.evaluate(test_dataset)

54/54 [==============================] - 1110s 20s/step - loss: 4.6227 - accuracy: 0.0000e+00


[4.622711658477783, 0.0]